<h1 style="text-align: center;">Programa7.HoldOut</h1>

<p style="text-align: center;">16-11-2026</p>

<p style="text-align: center;">
    <img src="img/ITQ.png" width="500" alt="Logotipo ITQ">
</p>

# 7. Validación y evaluación: método Hold-Out

**Nombre:** Nikolay Misael De La Cruz Melendres  
**Asignatura:** Fundamentos de Machine Learning  
**Docente:** Elvis David Pachacama Cabezas  
**Link Repositorio:** [NikolayDlc/fundamentosMachine](https://github.com/NikolayDlc/fundamentosMachine)

## Objetivo de la práctica

Comprender cómo separar un dataset en **entrenamiento, validación y prueba (test)**, cómo estandarizar sus variables y cómo entrenar y comparar dos clasificadores: `DummyClassifier` (modelo de referencia) y `SVC` (clasificador de máquinas de vectores de soporte).

**Dataset:** Iris. Contiene 150 flores, cuatro características numéricas por flor y tres especies que funcionan como clases o etiquetas.

## 1. Importación de bibliotecas

Se importan las herramientas necesarias para trabajar con el dataset y los modelos:

- `numpy as np`: operaciones numéricas y conteo de valores únicos.
- `datasets`: proporciona el dataset de ejemplo Iris.
- `DummyClassifier`: crea un clasificador simple que sirve como referencia inicial.
- `train_test_split`: divide los datos en subconjuntos separados.
- `StandardScaler`: estandariza variables numéricas utilizando la media y la desviación estándar aprendidas de los datos utilizados en `fit`.

In [1]:
import numpy as np
from sklearn import datasets
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Cargar el dataset Iris

- `datasets.load_iris()` carga las observaciones del conjunto Iris en la variable `iris`.
- `iris.target_names` contiene los nombres de las tres especies.

**Diferencia esencial:** `iris.data` contiene las características numéricas de las flores (**X**), mientras que `iris.target` contiene sus especies reales (**y**), que queremos predecir.

In [2]:
# Carga de datos
iris = datasets.load_iris()
print(iris.target_names)

['setosa' 'versicolor' 'virginica']


## 3. Examinar la cantidad de datos y las clases

- `iris.data.shape[0]`: número de **filas** o ejemplos (150 flores).
- `iris.data.shape[1]`: número de **columnas predictoras** o atributos (4 medidas).
- `set(iris.target)`: muestra las clases distintas; en Iris son `0`, `1` y `2`.
- `np.unique(iris.target, return_counts=True)`: devuelve dos arreglos: los valores de clase y cuántas observaciones hay de cada una.
- `valores, ocurrencias = ...`: guarda esos dos arreglos para mostrarlos con `print`.

**¿Para qué sirve?** Para conocer el tamaño del problema y revisar cuántos ejemplos hay por clase antes de dividir el dataset.

In [3]:
# Mostrar características de la tabla de datos.
print("Tabla de datos: %d instancias y %d atributos" % (iris.data.shape[0], iris.data.shape[1]))
print("Valores de la clase:", set(iris.target))

# Cuantificamos el número de instancias que contiene el dataset por clase
valores, ocurrencias = np.unique(iris.target, return_counts=True)
print(valores, ocurrencias)

Tabla de datos: 150 instancias y 4 atributos
Valores de la clase: {np.int64(0), np.int64(1), np.int64(2)}
[0 1 2] [50 50 50]


## 4. Partición externa: reservar el test

`train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)` realiza una división aleatoria conjunta de las características (**X**) y etiquetas (**y**):

- `X_training`, `y_training`: datos y respuestas del **80 % inicial**, destinados a entrenamiento y posterior validación.
- `X_test`, `y_test`: características y respuestas del **20 % reservado** para la evaluación final.
- `test_size=0.2`: aparta el 20 % para test; con 150 flores equivale a **30** para test y **120** para el resto.
- `random_state=42`: fija una semilla para que la partición aleatoria pueda reproducirse. El número 42 no mejora por sí solo la precisión.
- `np.unique(y_test, return_counts=True)`: cuenta las clases presentes en el test.

In [4]:
# Test: hold-out split 80-20%. # Partición externa
X_training, X_test, y_training, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
valores_test, ocur_test = np.unique(y_test, return_counts=True)
print('Test: ', 'clases:', valores_test, ' ocurrencias: ', ocur_test)

Test:  clases: [0 1 2]  ocurrencias:  [10  9 11]


## 5. Preparación para la estandarización

Primero se separan los conjuntos de entrenamiento, validación y test. Después se utiliza `StandardScaler` para calcular la media y la desviación estándar exclusivamente con `X_train`. Los conjuntos `X_val` y `X_test` se transforman utilizando los mismos parámetros, sin volver a ajustar el escalador. Esto evita utilizar información de validación y test durante el ajuste de la transformación.


In [5]:
# La estandarización se realizará después
# de separar entrenamiento, validación y test.

## 6. Partición interna y estandarización de los datos

Una vez reservado el test externo, se divide el conjunto inicial de 120 flores:

- `X_train`, `y_train`: características y etiquetas que se utilizan para **ajustar** el modelo.
- `X_val`, `y_val`: datos y respuestas reservados para **validar** su rendimiento.
- `test_size=0.2`: reserva el 20 % de los **120 datos restantes**, no el 20 % del dataset original.
- `np.unique(..., return_counts=True)`: revisa cuántos ejemplos de cada clase quedan en los conjuntos de entrenamiento y validación.

**Cantidades finales:** 96 flores para entrenamiento (64 % del total), 24 para validación (16 %) y 30 para test (20 %).

In [6]:
# Validación: hold-out split 80-20%. # Partición interna
X_train, X_val, y_train, y_val = train_test_split(X_training, y_training, test_size=0.2, random_state=42)

# Creamos el escalador
standardizer = StandardScaler()

# Aprendemos media y desviación estándar
# únicamente con los datos de entrenamiento
X_train = standardizer.fit_transform(X_train)

# Aplicamos la misma transformación
# a los datos de validación
X_val = standardizer.transform(X_val)

# Aplicamos la misma transformación
# a los datos de test
X_test = standardizer.transform(X_test)

print("Estandarización completada.")

valores_train, ocur_train = np.unique(y_train, return_counts=True)
print('Entrenamiento: ', ' clases:', valores_train, '  ocurrencias:', ocur_train)

valores_val, ocur_val = np.unique(y_val, return_counts=True)
print('Validation:    ', ' clases:', valores_val, '  ocurrencias:', ocur_val)

Estandarización completada.
Entrenamiento:   clases: [0 1 2]   ocurrencias: [32 30 34]
Validation:      clases: [0 1 2]   ocurrencias: [ 8 11  5]


### Estandarización después de la partición

Una vez separados los conjuntos de entrenamiento, validación y test,
se aplica StandardScaler para estandarizar las características. El escalador calcula la media y la desviación estándar utilizando únicamente los datos de entrenamiento (`X_train`). Posteriormente, los conjuntos de validación (`X_val`) y test
(`X_test`) se transforman utilizando esos mismos parámetros. Este procedimiento evita utilizar información de validación o test durante el ajuste del escalador y permite evaluar el modelo con datos independientes.

## 7. Crear un clasificador de referencia

`DummyClassifier(strategy='prior', random_state=42)` define un modelo base con el que se pueden comparar métodos más elaborados.

- `strategy='prior'`: el clasificador aprende la proporción de cada clase y, al predecir una etiqueta, elige la **clase más frecuente** en su entrenamiento.
- `random_state=42`: se indica en el código, pero **no afecta** las predicciones deterministas de la estrategia `prior`.
- `clf`: variable que almacena el objeto clasificador.

**¿Por qué se utiliza?** Porque permite comprobar si un modelo más avanzado aporta valor frente a una estrategia muy simple. Crear `clf` todavía no entrena el clasificador.

In [7]:
# Construcción del objeto que contiene el algoritmo de aprendizaje.
clf = DummyClassifier(strategy='prior', random_state=42)

## 8. Entrenar el clasificador

`clf.fit(X_train, y_train)` realiza el **entrenamiento**:

- `X_train`: características de las flores utilizadas para aprender.
- `y_train`: etiquetas correctas de esas flores.
- `fit(...)`: ajusta el clasificador usando esos datos.
- `clf = clf.fit(...)`: guarda en `clf` el objeto ya ajustado.

In [8]:
# Entrenamiento del algoritmo de aprendizaje.
clf = clf.fit(X_train, y_train)

## 9. Evaluar la exactitud (*accuracy*)

- `clf.score(X_val, y_val)`: produce la **exactitud** del clasificador en el conjunto de validación.
- `clf.score(X_test, y_test)`: obtiene la exactitud sobre el test externo.
- `val_accuracy` y `test_accuracy`: almacenan los resultados de esas evaluaciones.

**¿Qué significa exactitud?** Es la proporción de predicciones correctas respecto del total de ejemplos evaluados. Por ejemplo, 18 aciertos de 24 ejemplos equivalen a 0,75, es decir, 75 %.

In [9]:
# Evaluación del algoritmo de aprendizaje con el método "score" que devuelve directamente la métrica de 'accuracy'
val_accuracy = clf.score(X_val, y_val)
print("Exactitud en validación: ", val_accuracy)

test_accuracy = clf.score(X_test, y_test)
print("Exactitud en test: ", test_accuracy)

Exactitud en validación:  0.20833333333333334
Exactitud en test:  0.36666666666666664


## 10. Comparar predicciones con las etiquetas reales

- `clf.predict(X_val)`: genera las etiquetas **predichas** para las observaciones de validación.
- `y_val`: contiene las etiquetas **reales** de esas observaciones.
- `clf.predict(X_test)`: genera las predicciones del test externo.
- `y_test`: contiene las etiquetas reales del test.

In [10]:
# Predicciones del conjunto de validación
y_pred_val = clf.predict(X_val)

print("Predicciones de validación:", y_pred_val)
print("Etiquetas reales de validación:", y_val)

# Predicciones del conjunto de test
y_pred_test = clf.predict(X_test)

print("\nPredicciones de test:", y_pred_test)
print("Etiquetas reales de test:", y_test)

Predicciones de validación: [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales de validación: [1 1 0 0 0 2 1 2 2 2 1 1 1 1 1 0 2 0 1 0 1 1 0 0]

Predicciones de test: [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales de test: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


## 11. Entrenar y evaluar un modelo SVC

`SVC` es un clasificador de **máquinas de vectores de soporte** (*Support Vector Classification*). Aquí se utiliza como alternativa más elaborada que el modelo de referencia.

- `from sklearn.svm import SVC`: importa el algoritmo.
- `svc = SVC(C=0.5)`: crea el clasificador; `C` controla la penalización de errores y participa en el equilibrio entre ajuste y regularización.
- `svc.fit(X_train, y_train)`: lo entrena con los datos y etiquetas de entrenamiento.
- `svc.score(X_val, y_val)`: evalúa su exactitud en validación.
- `svc.score(X_test, y_test)`: calcula su exactitud sobre test.
- `np.round(val_accuracy*100, 4)`: convierte una proporción a porcentaje y redondea a cuatro decimales.

**¿Para qué sirve la comparación?** Para observar si el modelo `SVC` supera al clasificador simple `DummyClassifier`.

In [11]:
# Aplicamos un ejemplo con un clasificador más complejo que el "dummyclassifier"
from sklearn.svm import SVC
svc = SVC(C=0.5) # Definimos algoritmo
svc.fit(X_train, y_train) # Entrenamos modelo

val_accuracy = svc.score(X_val, y_val) # Evaluamos modelo en validación
print('Exactitud en validación ', np.round(val_accuracy*100, 4), '%')

test_accuracy = svc.score(X_test, y_test) # Evaluamos modelo en test
print('Exactitud en test ', np.round(test_accuracy*100, 4), '%')

Exactitud en validación  91.6667 %
Exactitud en test  96.6667 %


## 12. Guardar y recuperar un modelo con `pickle`

El módulo `pickle` permite serializar un objeto Python para guardarlo en un archivo y cargarlo posteriormente.

- `open('../models/model.pickle', 'wb')`: abre un archivo para **escribir datos binarios**.
- `pickle.dump(objeto, fw)`: guarda el objeto en el archivo.
- `open('../models/model.pickle', 'rb')`: abre el archivo para **leer datos binarios**.
- `pickle.load(fr)`: reconstruye el objeto guardado.

In [12]:
# Importamos las bibliotecas necesarias
import pickle
from pathlib import Path

# Creamos la carpeta models si no existe
Path('../models').mkdir(parents=True, exist_ok=True)

# Guardamos el modelo SVC que entrenamos anteriormente
with open('../models/model.pickle', 'wb') as fw:
    pickle.dump(svc, fw)

print("Modelo guardado correctamente.")

# Cargamos nuevamente el modelo
with open('../models/model.pickle', 'rb') as fr:
    modelo_cargado = pickle.load(fr)

print("Modelo cargado correctamente.")

Modelo guardado correctamente.
Modelo cargado correctamente.


In [13]:
# Comprobamos el modelo recuperado

accuracy_cargado = modelo_cargado.score(X_val, y_val)

print("Exactitud del modelo cargado:",
      np.round(accuracy_cargado * 100, 4), "%")

# Obtenemos predicciones
predicciones_cargadas = modelo_cargado.predict(X_val)

print("Predicciones:", predicciones_cargadas)
print("Valores reales:", y_val)


Exactitud del modelo cargado: 91.6667 %
Predicciones: [1 1 0 0 0 2 2 2 2 2 1 2 1 1 1 0 2 0 1 0 1 1 0 0]
Valores reales: [1 1 0 0 0 2 1 2 2 2 1 1 1 1 1 0 2 0 1 0 1 1 0 0]
